Attempting to see if all of Lilly's functions can be run in the new native python environment, and then attempt to import them to the new one

These all come from `patient_data.py` script:

In [1]:
import os
import pandas as pd
from exe_functions import build_path
import sys


#This appears to be the the function required for empty data above
#to generate an empty file
def new_empty_pt_data():
    # same pathing change has been added here
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_PatientData"), "empty_start.csv")
    date_cols = ["start_date", "censor_date"]
    pt_data = pd.read_csv(fp, sep=',', header=0, parse_dates=date_cols)
    return pt_data

In [ ]:
import os
import pandas as pd
from exe_functions import build_path
import sys

def import_pt_data(run_time):
    
    #It's always looking for YESTERDAY's data for the
    #patient.csv
    import_date = (run_time - pd.Timedelta("1 day")).date()
    
    #The path is relative to the environment location 
    # I have changed it to use the current directory where the Git
    # repo is downloaded but this may need to be adjusted
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_PatientData"), str(import_date) + "_pt_data.csv")
    date_cols = ["start_date", "censor_date"]
    try:
        pt_data = pd.read_csv(fp, sep=',', parse_dates=date_cols)
    except FileNotFoundError:
        while True:
            first_day = input("\nIs today the trial initiation?\n" 
                      + "If today is the first day, type 'yes' then hit Enter.\n"
                      + "Otherwise type 'no' then hit Enter.\n"
                      + "Answer here: ").lower()
            if first_day in ["yes", "no"]:
                first_day = first_day == "yes"
                break
            else:
                print("Input was not 'yes' or 'no'. Please try again.")
        if not first_day:
            input("\n" + str(import_date) + "_pt_data.csv in the 000_PatientData folder not found.\n"
                  + "This file should always exist with yesterday's date in the name. Please contact Lily.\n"
                  + "Press Enter to exit the program and close this window.")
            sys.exit()
        pt_data = new_empty_pt_data()
        return pt_data
    return pt_data

In [ ]:
import os
import pandas as pd
from exe_functions import build_path
import sys

def get_study_ids(pt_data):
    try:
    # Subsets the firstname column to find the unique study_id's available in the Pillsy data to update adherence
        study_ids_df = pt_data['record_id'].copy()
        unique_study_ids_df = study_ids_df.drop_duplicates()
        unique_study_ids_list = unique_study_ids_df.values.tolist()
    except ValueError:
        unique_study_ids_list = []
    except TypeError:
        unique_study_ids_list = []
        
    return unique_study_ids_list

These functions are from the `control_disconnect.py` script.

In [1]:
import pandas as pd
import numpy as np
import sys
import os
import re
import gc
import time
from datetime import datetime, date, timedelta
#import pytz
from pillsy_parser import identify_drug_freq, find_patient_rewards, get_drugName_list, find_taken_events, find_rewards
from patient_data import get_study_ids, new_empty_pt_data
from exe_functions import build_path
from driverRank import shift_t0_t1_rank_ids
from redcap_parser import update_pt_data_with_redcap

#This is basically identical to import_pt_data
def import_pt_data_control(run_time):
    import_date = (run_time - pd.Timedelta("1 day")).date()
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_PatientDataControl"), str(import_date) + "_pt_data.csv")
    date_cols = ["start_date", "censor_date"]
    try:
        pt_data = pd.read_csv(fp, sep=',', parse_dates=date_cols)
    except FileNotFoundError as fnfe:
        while True:
            first_day = input("\nIs today the trial initiation?\n" 
                      + "If today is the first day, type 'yes' then hit Enter.\n"
                      + "Otherwise type 'no' then hit Enter.\n"
                      + "Answer here: ").lower()
            if first_day in ["yes", "no"]:
                first_day = first_day == "yes"
                break
            else:
                print("Input was not 'yes' or 'no'. Please try again.")
        if not first_day:
            input("\n" + str(import_date) + "_pt_data_control.csv not found in the 000_PatientDataControl folder.\n"
                  + "This file should always exist with yesterday's date in the name. Please contact Lily.\n"
                  + "Press Enter to exit the program and close this window.")
            sys.exit()
        pt_data = new_empty_pt_data()
        return pt_data
    return pt_data

#Not sure redcap is necessary (but this may be the control file)
#Will need to check
def import_redcap_control(run_time):
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_REDCapControl"), str(run_time.date()) + "_redcap_control.csv")
    date_cols = ["start_date"]
    try:
        redcap = pd.read_csv(fp, sep=',', parse_dates=date_cols)
    except FileNotFoundError:
        input("\n" + str(run_time.date()) + "_redcap_control.csv was not found in the REDCapControl folder.\n"
              + "This should be today's date in YYYY-MM-DD format followed by _redcap_control.csv\n"
              + "and this must be placed in the 000_REDCapControl folder.\n"
              + "Please make sure this data has been downloaded and named properly.\n"
              + "Please run the program again after fixing the file name.\n"
              + "Press Enter to exit the program and close this window.")
        sys.exit()
    return redcap

#NEEDS REVIEW
#This appears to serve a different purpose altogether
#requires the pillsy input to run so may not be necessary
def check_control_disconnectedness(pillsy, redcap_data, pt_data, run_time):
    if not pt_data.empty and pillsy is not None:
        pt_data = find_rewards(pillsy, pt_data, run_time)
    
    pt_data = update_pt_data_with_redcap(redcap_data, pt_data, run_time)
    
    ranked_pt_data = new_empty_pt_data()
    for index, patient in pt_data.iterrows():
        if patient["censor"] != 1 and patient["censor_date"] > run_time.date():
            patient= shift_t0_t1_rank_ids(patient)
            patient["trial_day_counter"] += 1
            ranked_pt_data = ranked_pt_data.append(patient)
            
   
    ranked_pt_data.to_csv(build_path("000_PatientDataControl", str(run_time.date()) + "_pt_data_control.csv"),   index=False)

These functions are from the `pillsy_parser.py` script
Appears to rely on the existence of a `000_Pillsy` folder, which does not exist in the `CleverCap` archive and won't work locally because of that.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import re
import gc
import time
from datetime import datetime, date, timedelta
#import pytz

from patient_data import get_study_ids, new_empty_pt_data
from exe_functions import build_path


def import_Pillsy(run_time):
    """Import Pillsy pill taking history as pd.DataFrame from CSV
    """
    import_date = (run_time - pd.Timedelta("1 day")).date()
    pillsy_filename = str(import_date) + "_pillsy.csv"
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_Pillsy"), pillsy_filename)

    try:
        pillsy = pd.read_csv(fp)
    except FileNotFoundError:
        while True:
            first_day = input("\nIs today the trial initiation?\n" 
                      + "If today is the first day, type 'yes' then hit Enter.\n"
                      + "Otherwise type 'no' then hit Enter.\n"
                      + "Answer here: ").lower()
            if first_day in ["yes", "no"]:
                first_day = first_day == "yes"
                break
            else:
                print("Input was not 'yes' or 'no'. Please try again.")
        if not first_day:
            input("\n" + str(import_date) + "_pillsy.csv was not found in the 000_Pillsy folder.\n"
                  + "This should be yesterday's date in YYYY-MM-DD format followed by _pillsy.csv\n"
                  + "and this must be placed in the Pillsy folder.\n"
                  + "Please make sure this data has been downloaded and named properly.\n"
                  + "Please run the program again after fixing the file name.\n"
                  + "Press Enter to exit the program and close this window.")
            sys.exit()
        return None
    
    tz_ref = {
        "HDT": "-0900",
        "HST": "-1000",
        "AKDT": "-0800",
        "AKST": "-0900",
        "PDT": "-0700",
        "PST": "-0800",
        "MDT": "-0600",
        "MST": "-0700",
        "CDT": "-0500",
        "CST": "-0600",
        "EDT": "-0400",
        "EST": "-0500"
    }

    def converter(time_string):
        import re
        tz_abbr = re.search(r"\d\d:\d\d .M ([A-Z]{2,4}) \d{4}-\d\d-\d\d", time_string).group(1)
        return time_string.replace(tz_abbr, tz_ref.get(tz_abbr, "-0500"))

    pillsy.dropna(
    axis=0,
    how='all',
    thresh=None,
    subset=None,
    inplace=True)
    #https://hackersandslackers.com/pandas-dataframe-drop/
    
    pillsy["eventTime"] = pd.to_datetime(pd.Series([converter(str_dt) for str_dt in pillsy["eventTime"]])) #, utc=True)
    # Note: In this dataset our study_id is actually 'firstname', hence the drop of patientId
    # Note: firstname is currently read in as int64 dtype
    pillsy.drop(["patientId", "lastname", "method", "platform"], axis=1, inplace=True)
    return pillsy


def get_drugName_list(patient_entries):
    try:
        drugNames_df = patient_entries['drugName']
        unique_drugNames_df = drugNames_df.drop_duplicates()
        unique_drugNames_df_list = unique_drugNames_df.values.tolist()
    except ValueError:
        unique_drugNames_df_list = []
    return unique_drugNames_df_list



def identify_drug_freq(drugName):
    """
    identify_drug_freq function checks the drugName for QD or BID to return either 1 or 2
    as the underlying expected number of taken events
    :param drugName:
    :return: drugFreq
    """
    drugFreq = 0
    # drugName is a String
    # .find returns -1 if it doesn't find the String QD or BID in the drugName String
    # If the drugName does contain QD or BID, then .find() will return an int > -1 (0 or more)
    # at the index of the first occurrence of the BID or QD string
    if drugName.find('QD') > -1:
        drugFreq = 1
    elif drugName.find('BID') > -1:
        drugFreq = 2
    # Returns the number of doses that the given Medication is
    return drugFreq


def find_taken_events(drug, drug_subset):
    """
    find_taken_events function finds the adherence for a particular drug when run over a particular time period using
    the investigator specified algorithm for evaluating OPEN/CLOSE sequences as taken events
    (Note: 
    Ways to identify Taken Events:
        1. OPEN 
        2. CLOSE CLOSE (within 15 min of each other)
        3. CLOSE OPEN  (within 15 min of each other)
        4. if drug_freq > 1, then for second taken event only need OPEN/CLOSE with wait time of 2 hr 45 min after first
    After identifying first taken event, need to wait 2 hr 45 min to identify second taken event if drug_freq == 2
    :param drug:
    :param drug_subset:
    :return:
    """
#     fifteen_min = pd.Timedelta('15 minutes')
#     two_hr_45_min = pd.Timedelta('2 hours, 45 minutes')
    drug_freq = identify_drug_freq(drug)
    taken = 0
    last_event = None
    waiting_after_close = False
    for index, event in drug_subset.iterrows():
        if last_event is None:
            if drug_freq == 1:
                return 1.0
            taken += 1
            last_event = event
        else:
            if last_event['eventTime'] + pd.Timedelta('2 hours, 45 minutes') < event['eventTime']:
                    return 1.0 # must be second taken event
    return 0.5

def compute_taken_over_expected(patient, timeframe_pillsy_subset, num_pillsy_meds):
    """
    compute_taken_over_expected function runs find_taken_events for each drug to find the adherence of each drug
    over a given time frame and then computes the average adherence for the time period by dividing the sum of
    adherence of all drugs over the number of drugs (i.e. expected maximum sum of adherence) for a patient
    :param patient:
    :param timeframe_pillsy_subset:
    :return:
    """
    timeframe_drugs = get_drugName_list(timeframe_pillsy_subset)
    print("timeframe_drugs:", timeframe_drugs)
    timeframe_adherence_by_drug = []
    for drug in timeframe_drugs:
        drug_subset = timeframe_pillsy_subset[timeframe_pillsy_subset['drugName'] == drug]
        this_drug_adherence = find_taken_events(drug, drug_subset)
        timeframe_adherence_by_drug.append(this_drug_adherence)
    if not timeframe_pillsy_subset.empty and num_pillsy_meds > 0:
        sum_timeframe_adherence = sum(timeframe_adherence_by_drug)
        taken_over_expected = sum_timeframe_adherence / num_pillsy_meds
        print("timeframe_adherence_by_drug:", timeframe_adherence_by_drug)
        print("sum_timeframe_adherence:", sum_timeframe_adherence)
        print("num_pillsy_meds:", num_pillsy_meds)
        print("taken_over_expected:", taken_over_expected)
    else:
        taken_over_expected = 0
    return taken_over_expected



def calc_avg_adherence(patient):
    """
    calc_avg_adherence function updates the avg adherences at days 1,3,7 with updated shifted daily adherence values
    :param patient:
    :return:
    """
    if patient["trial_day_counter"] < 3:
        patient["avg_adherence_1day"] = patient["adherence_day1"]
    elif 3 <= patient["trial_day_counter"] < 7:
        patient["avg_adherence_3day"] = (patient["adherence_day1"] + patient["adherence_day2"] + patient["adherence_day3"]) / 3
        patient["avg_adherence_1day"] = patient["adherence_day1"]
    elif patient["trial_day_counter"] >= 7:
        patient["avg_adherence_7day"] = (
                                          patient["adherence_day1"] + patient["adherence_day2"] + patient["adherence_day3"] + patient["adherence_day4"] + patient["adherence_day5"] + patient["adherence_day6"] + patient["adherence_day7"]) / 7
        patient["avg_adherence_3day"] = (patient["adherence_day1"] + patient["adherence_day2"] + patient["adherence_day3"]) / 3
        patient["avg_adherence_1day"] = patient["adherence_day1"]
    return patient

  

def find_patient_rewards(pillsy_subset, patient, run_time):
    # From pillsy_subset, get timezone of patient, then use that to calculate the time cut points so that it is relative to the
    # patient's view of time.
    curtime = run_time
    tzinfo_first_row = run_time.tzinfo
    first_row = pillsy_subset.head(1)
    if not first_row.empty:
        #print(first_row['eventTime'].values[0])
        tzinfo_first_row = first_row['eventTime'].values[0].tzinfo
        curtime = curtime.astimezone(tzinfo_first_row)
        
    seven_day_ago = (curtime - timedelta(days=7)).date()
    seven_day_ago_12am = datetime.combine(seven_day_ago, datetime.min.time()).astimezone(tzinfo_first_row)    
    
    six_day_ago = (curtime - timedelta(days=6)).date()
    six_day_ago_12am = datetime.combine(six_day_ago, datetime.min.time()).astimezone(tzinfo_first_row)    
    
    five_day_ago = (curtime - timedelta(days=5)).date()
    five_day_ago_12am = datetime.combine(five_day_ago, datetime.min.time()).astimezone(tzinfo_first_row)
        
    four_day_ago = (curtime - timedelta(days=4)).date()
    four_day_ago_12am = datetime.combine(four_day_ago, datetime.min.time()).astimezone(tzinfo_first_row)
    
    three_day_ago = (curtime - timedelta(days=3)).date()
    three_day_ago_12am = datetime.combine(three_day_ago, datetime.min.time()).astimezone(tzinfo_first_row)
    
    two_day_ago = (curtime - timedelta(days=2)).date()
    two_day_ago_12am = datetime.combine(two_day_ago, datetime.min.time()).astimezone(tzinfo_first_row)
    
    yesterday = (curtime - timedelta(days=1)).date()
    yesterday_12am =datetime.combine(yesterday, datetime.min.time()).astimezone(tzinfo_first_row)
    
    today_current_time = curtime
        # https://www.w3resource.com/python-exercises/date-time-exercise/python-date-time-exercise-8.php
    today_12am = datetime.combine(today_current_time, datetime.min.time()).astimezone(tzinfo_first_row)
        
    pillsy_seven_day_ago_subset = pillsy_subset[pillsy_subset["eventTime"] < six_day_ago_12am]
    pillsy_seven_day_ago_subset = pillsy_seven_day_ago_subset[pillsy_seven_day_ago_subset["eventTime"] >= seven_day_ago_12am]

    pillsy_six_day_ago_subset = pillsy_subset[pillsy_subset["eventTime"] < five_day_ago_12am]
    pillsy_six_day_ago_subset = pillsy_six_day_ago_subset[pillsy_six_day_ago_subset["eventTime"] >= six_day_ago_12am]

    pillsy_five_day_ago_subset = pillsy_subset[pillsy_subset["eventTime"] < four_day_ago_12am]
    pillsy_five_day_ago_subset = pillsy_five_day_ago_subset[pillsy_five_day_ago_subset["eventTime"] >= five_day_ago_12am]

    pillsy_four_day_ago_subset = pillsy_subset[pillsy_subset["eventTime"] < three_day_ago_12am]
    pillsy_four_day_ago_subset = pillsy_four_day_ago_subset[pillsy_four_day_ago_subset["eventTime"] >= four_day_ago_12am]

    pillsy_three_day_ago_subset = pillsy_subset[pillsy_subset["eventTime"] < two_day_ago_12am]
    pillsy_three_day_ago_subset = pillsy_three_day_ago_subset[pillsy_three_day_ago_subset["eventTime"] >= three_day_ago_12am]

    pillsy_two_day_ago_subset = pillsy_subset[pillsy_subset["eventTime"] < yesterday_12am]
    pillsy_two_day_ago_subset = pillsy_two_day_ago_subset[pillsy_two_day_ago_subset["eventTime"] >= two_day_ago_12am]

    pillsy_yesterday_subset = pillsy_subset[pillsy_subset["eventTime"] < today_12am]
    pillsy_yesterday_subset = pillsy_yesterday_subset[pillsy_yesterday_subset["eventTime"] >= yesterday_12am]

    pillsy_early_today_subset = pillsy_subset[pillsy_subset["eventTime"] >= today_12am]
    pillsy_early_today_subset = pillsy_early_today_subset[pillsy_early_today_subset["eventTime"] < today_current_time]
    
    pillsy_yesterday_disconnectedness_subset = pillsy_subset[pillsy_subset["eventTime"] < today_current_time]
    pillsy_yesterday_disconnectedness_subset = pillsy_yesterday_subset[pillsy_yesterday_subset["eventTime"] >= yesterday_12am]
    
    print("\nBEGIN CHECKING REWARDS FOR PT " + str(patient["record_id"]) + "\n")
    
    print("Computing... reward_value_t0 in pillsy_yesterday_subset with # med:", patient["num_pillsy_meds_t0"])
    reward_value_t0 = compute_taken_over_expected(patient, pillsy_yesterday_subset, patient["num_pillsy_meds_t0"])
    print("reward_value_t0:", reward_value_t0)
    print("Computing... reward_value_t1 in pillsy_two_day_ago_subset with # med:", patient["num_pillsy_meds_t1"])
    reward_value_t1 = compute_taken_over_expected(patient, pillsy_two_day_ago_subset, patient["num_pillsy_meds_t1"])
    print("reward_value_t1:", reward_value_t1)

    adherence_day3 = compute_taken_over_expected(patient, pillsy_three_day_ago_subset, patient["num_pillsy_meds_t2"])
    adherence_day4 = compute_taken_over_expected(patient, pillsy_four_day_ago_subset, patient["num_pillsy_meds_t3"])
    adherence_day5 = compute_taken_over_expected(patient, pillsy_five_day_ago_subset, patient["num_pillsy_meds_t4"])
    adherence_day6 = compute_taken_over_expected(patient, pillsy_six_day_ago_subset, patient["num_pillsy_meds_t5"])
    adherence_day7 = compute_taken_over_expected(patient, pillsy_seven_day_ago_subset, patient["num_pillsy_meds_t6"])
   

    print("Computing... early_rx_use in pillsy_early_today_subset with # med:", patient["num_pillsy_meds_t0"])
    early_rx_use = compute_taken_over_expected(patient, pillsy_early_today_subset, patient["num_pillsy_meds_t0"])
    print("early_rx_use:", early_rx_use)
    
    print("Computing... yesterday_disconnectedness in pillsy_yesterday_disconnectedness_subset with # med:", patient["num_pillsy_meds_t0"])
    observed_num_drugs = get_drugName_list(pillsy_yesterday_disconnectedness_subset)
    print("expected # rx:",  patient["num_pillsy_meds_t0"],"// observed # rx:", len(observed_num_drugs), "\n observed rx names:",observed_num_drugs)
    
    # if we didn't send reward for 2 days ago yesterday, we send it today
    if patient["flag_send_reward_value_t1"] == False:
        patient["flag_send_reward_value_t1"] = True   
    else:
        patient["flag_send_reward_value_t1"] == False
        
    # now we check to see if we should send yesterday's reward today 
    if len(observed_num_drugs) == patient["num_pillsy_meds_t0"]:
        yesterday_disconnectedness = 1
        # patient is connected so we can send the reward for yesterday
        patient["flag_send_reward_value_t0"] = True
    else:
        # patient is disconnected so we will give it some time to back fill and send tomorrow
        yesterday_disconnectedness = -1
        patient["num_dates_disconnectedness"] += 1
        patient["flag_send_reward_value_t0"] = False   
    print("disconnectedness:", yesterday_disconnectedness)

    # Update data frame with new values for reward and
    patient["reward_value_t0"] = reward_value_t0
    patient["reward_value_t1"] = reward_value_t1
    patient["early_rx_use"] = early_rx_use
    
    if early_rx_use > 0:
        patient["num_dates_early_rx_use"] += 1

    patient["adherence_day7"] = adherence_day7
    patient["adherence_day6"] = adherence_day6
    patient["adherence_day5"] = adherence_day5
    patient["adherence_day4"] = adherence_day4
    patient["adherence_day3"] = adherence_day3
    patient["adherence_day2"] = reward_value_t1
    patient["adherence_day1"] = reward_value_t0
    
    patient["dichot_adherence_day7"] = (adherence_day7 > 0)*1
    patient["dichot_adherence_day6"] = (adherence_day6 > 0)*1
    patient["dichot_adherence_day5"] = (adherence_day5 > 0)*1
    patient["dichot_adherence_day4"] = (adherence_day4 > 0)*1
    patient["dichot_adherence_day3"] = (adherence_day3 > 0)*1
    patient["dichot_adherence_day2"] = (reward_value_t1 > 0)*1
    patient["dichot_adherence_day1"] = (reward_value_t0 > 0)*1


    patient["total_dichot_adherence_past7"] = (patient["dichot_adherence_day7"] +
    patient["dichot_adherence_day6"] +
    patient["dichot_adherence_day5"] +
    patient["dichot_adherence_day4"] +
    patient["dichot_adherence_day3"] +
    patient["dichot_adherence_day2"] +
    patient["dichot_adherence_day1"] )
    
    if patient["disconnectedness"] == -1 and yesterday_disconnectedness == -1:
        patient["num_days_continuously_disconnected"]  += 1
    else:
        patient["num_days_continuously_disconnected"]  = 0
        
    if patient["num_days_continuously_disconnected"] > 6:
        patient["contact_disconnected"] = True
    else:
        patient["contact_disconnected"] = False

    patient["disconnectedness"] = yesterday_disconnectedness
   
    # We update the avg adherences at days 1,3,7 with updated shifted daily adherence values:
    patient = calc_avg_adherence(patient)

    return patient 

def find_rewards(pillsy, pt_data, run_time):
    """
    find_rewards function iteratees through the patients in pt_data to update their adherence measurements
    (daily and cumulative) and thereby reward values; also updates diconnectedness and early_rx_use features
    :param pillsy:
    :param pillsy_study_ids_list:
    :param pt_data:
    :param run_time:
    :return:
    """
    rewarded_pt_data = new_empty_pt_data()
    study_ids_list = get_study_ids(pt_data)
    for study_id in study_ids_list:
        # Filter by firstname = study_id to get data for just this one patient
        patient_pillsy_subset = pillsy[pillsy["firstname"] == study_id]
        patient_row = pt_data[pt_data["record_id"] == study_id].iloc[0]
        # This function will update the patient attributes with the updated adherence data that we will find from pillsy
        patient_row = find_patient_rewards(patient_pillsy_subset, patient_row, run_time)
        rewarded_pt_data = rewarded_pt_data.append(patient_row)
    return rewarded_pt_data

All of the following functions are drawn from the `redcap_parser.py` script

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta
from dateutil import parser, tz
import pickle
import os
import re
import sys
from patient_data import new_empty_pt_data

from exe_functions import build_path

# For a DataFrame a dict can specify that different values should be replaced in different columns. For example, {'a': 1, 'b': 'z'} looks for the value 1 in column ‘a’ and the value ‘z’ in column ‘b’ and replaces these values with whatever is specified in value. The value parameter should not be None in this case. You can treat this as a special case of passing two lists except that you are specifying the column to search in.

def redcap_vars_converter(redcap_df):
    # Tested
    redcap_df = redcap_df.replace({'age': 1}, "18-34")
    redcap_df = redcap_df.replace({'age': 2}, "35-44")
    redcap_df = redcap_df.replace({'age': 3}, "45-54")
    redcap_df = redcap_df.replace({'age': 4}, "55-64")
    redcap_df = redcap_df.replace({'age': 5}, "65-74")
    redcap_df = redcap_df.replace({'age': 6}, "75-84")
    redcap_df = redcap_df.replace({'sex': 1}, "F")
    redcap_df = redcap_df.replace({'sex': 2}, "M")
    redcap_df = redcap_df.replace({'sex': 0}, "Not listed")
    redcap_df = redcap_df.replace({'num_years_dm_rx': 1}, "0")
    redcap_df = redcap_df.replace({'num_years_dm_rx': 2}, "1-2")
    redcap_df = redcap_df.replace({'num_years_dm_rx': 3}, "3-4")  # TODO
    redcap_df = redcap_df.replace({'num_years_dm_rx': 4}, "5+")
    redcap_df = redcap_df.replace({'hba1c': 1}, "7.5-8.0")
    redcap_df = redcap_df.replace({'hba1c': 2}, "8.1-8.9")
    redcap_df = redcap_df.replace({'hba1c': 3}, "9.0-9.9")
    redcap_df = redcap_df.replace({'hba1c': 4}, "10+")
    redcap_df = redcap_df.replace({'num_physicians': 1}, "1")
    redcap_df = redcap_df.replace({'num_physicians': 2}, "2-3")
    redcap_df = redcap_df.replace({'num_physicians': 3}, "4+")  # fixed
    redcap_df = redcap_df.replace({'num_rx': 1}, "1")
    redcap_df = redcap_df.replace({'num_rx': 2}, "2-4")
    redcap_df = redcap_df.replace({'num_rx': 3}, "5-9")
    redcap_df = redcap_df.replace({'num_rx': 4}, "10+")
    redcap_df = redcap_df.replace({'automaticity': 0}, "0")
    redcap_df = redcap_df.replace({'automaticity': 1}, "1")
    redcap_df = redcap_df.replace({'automaticity': 2}, "2-3")
    redcap_df = redcap_df.replace({'automaticity': 3}, "4")
    redcap_df = redcap_df.replace({'pt_activation': 1}, "yes")
    redcap_df = redcap_df.replace({'pt_activation': 2}, "most")
    redcap_df = redcap_df.replace({'pt_activation': 3}, "no")
    redcap_df = redcap_df.replace({'reason_dm_rx': 1}, "Supposed to")
    redcap_df = redcap_df.replace({'reason_dm_rx': 2}, "Own good")
    redcap_df = redcap_df.replace({'reason_dm_rx': 3}, "No choice")
    redcap_df = redcap_df.replace({'reason_dm_rx': 4}, "Feel good")
    redcap_df = redcap_df.replace({'reason_dm_rx': 5}, "Important")
    redcap_df = redcap_df.replace({'non_adherence': 0}, "0")
    redcap_df = redcap_df.replace({'non_adherence': 1}, "1")
    redcap_df = redcap_df.replace({'non_adherence': 2}, "2-3")
    redcap_df = redcap_df.replace({'non_adherence': 3}, "4-6")
    redcap_df = redcap_df.replace({'non_adherence': 4}, "7+")
    redcap_df = redcap_df.replace({'edu_level': 1}, "HS or below/HS grad")
    redcap_df = redcap_df.replace({'edu_level': 2}, "Some college")
    redcap_df = redcap_df.replace({'edu_level': 3}, "College grad/Postgrad")
    redcap_df = redcap_df.replace({'edu_level': 4}, "other")
    redcap_df = redcap_df.replace({'employment_status': 1}, "Employed")
    redcap_df = redcap_df.replace({'employment_status': 2}, "Retired/Other")
    redcap_df = redcap_df.replace({'marital_status': 1}, "Married/partner")
    redcap_df = redcap_df.replace({'marital_status': 2}, "window/divorced/single/other")
    return redcap_df

def import_redcap(run_time):
    """Import REDCap patients that are enrolling on an ongoing basis as a pd.DataFrame from a CSV

    Does not overwrite old pt_data object since we calculate additional features based on historical data that should not be overwritten.
    This new data will be used to update existing patients: if censored, changes in pillsy rx
    """
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_REDCap"), str(run_time.date()) + "_redcap.csv")
    date_cols = ["start_date"]
    try:
        redcap = pd.read_csv(fp, sep=',', parse_dates=date_cols)
    except FileNotFoundError:
        input("\n" + str(run_time.date()) + "_redcap.csv was not found in the 000_REDCap folder.\n"
              + "This should be today's date in YYYY-MM-DD format followed by _redcap.csv\n"
              + "and this must be placed in the REDCap folder.\n"
              + "Please make sure this data has been downloaded and named properly.\n"
              + "Please run the program again after fixing the file name.\n"
              + "Press Enter to exit the program and close this window.")
        sys.exit()
    redcap = redcap_vars_converter(redcap)
    return redcap

def get_unique_study_ids(df):
    # Subsets the redcap data into the record_id column
    study_ids = df['record_id'].copy()
    # Drops duplicate id's so we have a succinct list (not particularly necessary)
    unique_study_ids = study_ids.drop_duplicates()
    # convert the record id's into a list to return
    unique_study_ids_list = unique_study_ids.values.tolist()
    # returns a list of the unique record_id's in the redcap data
    return unique_study_ids_list


##CHECK FOR COMPATABILITY
def update_pt_data_with_redcap(redcap_data, pt_data, run_time):
    """Update patient data with new and existing patients from REDCap, return pt_data.

    redcap_data -- pd.DataFrame, parsed from CSV downloaded from REDCap
    pt_data -- pd.DataFrame, parsed from / written to CSV each run
    run_time -- to be used for later testing

    Add any new patients to the pt_data object.
    Update censor and pillsy related variables for existing patients.
    """
    unique_study_ids_list_redcap = get_unique_study_ids(redcap_data)   
    unique_study_ids_list_pt_data = get_unique_study_ids(pt_data)
    new_pt_data = new_empty_pt_data()
    # Updating existing patients
    if not pt_data.empty:
        for index, patient in pt_data.iterrows():
            record_id = patient["record_id"]
            row = redcap_data[redcap_data["record_id"] == record_id]
            if not row.empty:
                row = row.iloc[0]
                # need to make sure via iterating it updates this data in the df
                patient["censor"] = row["censor"]
                patient["num_twice_daily_pillsy_meds"] = row["num_twice_daily_pillsy_meds"]
                patient["pillsy_meds_agi"] = row["pillsy_meds___1"]
                patient["pillsy_meds_dpp4"] = row["pillsy_meds___2"]
                patient["pillsy_meds_glp1"] = row["pillsy_meds___3"]
                patient["pillsy_meds_meglitinide"] = row["pillsy_meds___4"]
                patient["pillsy_meds_metformin"] = row["pillsy_meds___5"]
                patient["pillsy_meds_sglt2"] = row["pillsy_meds___6"]
                patient["pillsy_meds_sulfonylurea"] = row["pillsy_meds___7"]
                patient["pillsy_meds_thiazolidinedione"] = row["pillsy_meds___8"]
                # Shift previous number of meds over a day for measurement of backfilling of data 
                patient["num_pillsy_meds_t6"] = patient["num_pillsy_meds_t5"]
                patient["num_pillsy_meds_t5"] = patient["num_pillsy_meds_t4"]
                patient["num_pillsy_meds_t4"] = patient["num_pillsy_meds_t3"]
                patient["num_pillsy_meds_t3"] = patient["num_pillsy_meds_t2"]
                patient["num_pillsy_meds_t2"] = patient["num_pillsy_meds_t1"]
                patient["num_pillsy_meds_t1"] = patient["num_pillsy_meds_t0"]
                patient["num_pillsy_meds_t0"] = row["bottles"]
                new_pt_data = new_pt_data.append(patient)

    # Adding new patients
    for id in unique_study_ids_list_redcap:
        if id not in unique_study_ids_list_pt_data:

            redcap_row = redcap_data[redcap_data['record_id'] == id].iloc[0]
            if redcap_row['censor'] == 1:
                continue # skips this patient in the for loop and continues to the next patient
            if redcap_row['start_date'] > run_time.date(): 
                continue 
            if redcap_row["race___5"] == 1 or redcap_row["race___6"] == 1 or redcap_row["race___7"] == 1:
                race_other = 1
            else:
                race_other = 0
            censor_date = (redcap_row["start_date"] + timedelta(days=183)).date() ##TODO check trial length
            new_row = pd.Series({'record_id': id,
                                 'trial_day_counter': 0,
                                 'age': redcap_row["age"],
                                 'sex': redcap_row["sex"],
                                 'num_years_dm_rx': redcap_row["num_years_dm_rx"],
                                 'hba1c': redcap_row["hba1c"],
                                 'race_white': redcap_row["race___1"],
                                 'race_black': redcap_row["race___2"],
                                 'race_asian': redcap_row["race___3"],
                                 'race_hispanic': redcap_row["race___4"],
                                 'race_other': race_other,
                                 'num_physicians': redcap_row["num_physicians"],
                                 'num_rx': redcap_row["num_rx"],
                                 'concomitant_insulin_use': redcap_row["concomitant_insulin_use"],
                                 'automaticity': redcap_row["automaticity"],
                                 'pt_activation': redcap_row["pt_activation"],
                                 'reason_dm_rx': redcap_row["reason_dm_rx"],
                                 'non_adherence': redcap_row["non_adherence"],
                                 'edu_level': redcap_row["edu_level"],
                                 'employment_status': redcap_row["employment_status"],
                                 'marital_status': redcap_row["marital_status"],
                                 'pillsy_meds_agi': redcap_row["pillsy_meds___1"],
                                 'pillsy_meds_dpp4': redcap_row["pillsy_meds___2"],
                                 'pillsy_meds_glp1': redcap_row["pillsy_meds___3"],
                                 'pillsy_meds_meglitinide': redcap_row["pillsy_meds___4"],
                                 'pillsy_meds_metformin': redcap_row["pillsy_meds___5"],
                                 'pillsy_meds_sglt2': redcap_row["pillsy_meds___6"],
                                 'pillsy_meds_sulfonylurea': redcap_row["pillsy_meds___7"],
                                 'pillsy_meds_thiazolidinedione': redcap_row["pillsy_meds___8"],
                                 'num_pillsy_meds_t0': redcap_row["bottles"],
                                 'num_pillsy_meds_t1': 0,
                                 'num_pillsy_meds_t2': 0,
                                 'num_pillsy_meds_t3': 0,
                                 'num_pillsy_meds_t4': 0,
                                 'num_pillsy_meds_t5': 0,
                                 'num_pillsy_meds_t6': 0,
                                 'start_date': redcap_row["start_date"],
                                 'censor_date': censor_date,
                                 'num_twice_daily_pillsy_meds': redcap_row["num_twice_daily_pillsy_meds"],
                                 'censor': redcap_row["censor"],
                                 'num_day_since_no_sms': 0,
                                 'num_day_since_pos_framing':0,
                                 'num_day_since_neg_framing':0,
                                 'num_day_since_history':0,
                                 'num_day_since_social':0,
                                 'num_day_since_content':0,
                                 'num_day_since_reflective':0,
                                 'total_dichot_adherence_past7':0,
                                 'flag_send_reward_value_t0':False,
                                 'flag_send_reward_value_t1':False,
                                 'num_dates_disconnectedness': 0,
                                 'num_days_continuously_disconnected':0,
                                 'contact_disconnected':False,
                                 'num_dates_early_rx_use':0,}, name=id)
            new_pt_data = new_pt_data.append(new_row)

    return new_pt_data




All of the following are drawn from `driverRank.py`, one of the main features of the workflow.

In [ ]:
# <Dependencies>
from azure.cognitiveservices.personalizer import PersonalizerClient
from azure.cognitiveservices.personalizer.models import RankRequest
from msrest.authentication import CognitiveServicesCredentials
from Actions import get_framing_actions, get_history_actions, get_social_actions, get_content_actions, get_reflective_actions
from datetime import datetime, date, timedelta
#import pytz
import os
import pandas as pd

from exe_functions import build_path



def new_empty_rank_log(run_time):
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_RankData"), "empty_rank_log.csv")
    date_cols = ["start_date", "censor_date"]
    ranking_log = pd.read_csv(fp, sep=',', header=0, parse_dates=date_cols)
    return ranking_log

def write_rank_log(ranking_log, run_time):
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_RankData"), str(run_time.date()) + "_rank_log.csv")
    ranking_log.to_csv(fp, index=False)

def write_sms_history(pt_data, run_time):
    fp = build_path(os.path.abspath(os.curdir) + ("\\000_SMS_TO_SEND"), str(run_time.date()) + "_sms_history.csv")
    # Subset updated_pt_dict to what we need for reward calls and put in dataframe
    # create an Empty DataFrame object
    column_values = ['record_id','sms_msg_today', 'factor_set', 'text_number',  'trial_day_counter','censor_date', 'num_days_continuously_disconnected','contact_disconnected']
    sms_history_dataframe = pd.DataFrame(columns=column_values)

    for pt, data in pt_data.iterrows():
        # Reward value, Rank_Id's
        sms_history_dataframe.loc[len(sms_history_dataframe)] = [data["record_id"], data["sms_msg_today"], data["factor_set"], data["text_number"], data["trial_day_counter"], str(data["censor_date"]), data["num_days_continuously_disconnected"], data["contact_disconnected"]]
    date_cols = ["start_date", "censor_date"]
    control_fp = build_path(os.path.abspath(os.curdir) + ("\\000_PatientDataControl"), str(run_time.date()) + "_pt_data_control.csv")
    controls = pd.read_csv(control_fp, sep=',', header=0, parse_dates=date_cols)

    for pt, data in controls.iterrows():
        # Reward value, Rank_Id's
        sms_history_dataframe.loc[len(sms_history_dataframe)] = [data["record_id"], 'CONTROL', 'CONTROL', 'CONTROL', data["trial_day_counter"], str(data["censor_date"]), data["num_days_continuously_disconnected"], data["contact_disconnected"]]
    
    # Writes CSV for RA to send text messages.
    sms_history_dataframe.to_csv(fp, index=False)

def run_ranking(patient, client, run_time):
    """Send rank calls to Personalizer and update corresponding patient variables.

    1. Shift rank ids
    2. Make rank calls (framing, history, social, content, reflective)
        a. construct event_id
        b. get context features
        c. get actions
        d. call RankRequest
        e. get response, convert to int
        f. update patient var
    3. Update patient num days since rank calls
    4. Update appropriate sms vars in patient row
    """
    patient= shift_t0_t1_rank_ids(patient)
    # framing
    rank_id_framing = str(patient["record_id"]) + "_" + str(patient["trial_day_counter"]) + "_frame"
    patient["rank_id_framing_t0"] = rank_id_framing
    context = get_framing_context(patient)
    actions = get_framing_actions()

    frame_rank_request = RankRequest(actions=actions, context_features=context, event_id=rank_id_framing)
    frame_response = client.rank(rank_request=frame_rank_request)
    framing_ranked = frame_response.reward_action_id

    patient = update_framing_ranking(patient, framing_ranked)

    # history
    
    if patient["disconnectedness"] == 1 and patient["trial_day_counter"] > 7:
        rank_id_history = str(patient["record_id"]) + "_" + str(patient["trial_day_counter"]) + "_history"
        patient["rank_id_history_t0"] = rank_id_history
        context = get_history_context(patient)
        actions = get_history_actions()
        
        history_rank_request = RankRequest(actions=actions, context_features=context, event_id=rank_id_history)
        history_response = client.rank(rank_request=history_rank_request)
        history_ranked = history_response.reward_action_id
        history_response_flag = True

    else:
        rank_id_history = None
        patient["rank_id_history_t0"] = None
        history_ranked = "noHistory"
        history_response_flag = False
        
        
    patient = update_history_ranking(patient, history_ranked)

    # social
    rank_id_social = str(patient["record_id"]) + "_" + str(patient["trial_day_counter"]) + "_social"
    patient["rank_id_social_t0"] = rank_id_social
    context = get_social_context(patient)
    actions = get_social_actions()

    social_rank_request = RankRequest(actions=actions, context_features=context, event_id=rank_id_social)
    social_response = client.rank(rank_request=social_rank_request)
    social_ranked = social_response.reward_action_id

    patient = update_social_ranking(patient,social_ranked)

    # content
    rank_id_content = str(patient["record_id"]) + "_" + str(patient["trial_day_counter"]) + "_content"
    patient["rank_id_content_t0"] = rank_id_content
    context = get_content_context(patient)
    actions = get_content_actions()

    content_rank_request = RankRequest(actions=actions, context_features=context, event_id=rank_id_content)
    content_response = client.rank(rank_request=content_rank_request)
    content_ranked = content_response.reward_action_id

    patient = update_content_ranking(patient, content_ranked)

    # reflective
    rank_id_reflective = str(patient["record_id"]) + "_" + str(patient["trial_day_counter"]) + "_reflective"
    patient["rank_id_reflective_t0"] = rank_id_reflective
    context = get_reflective_context(patient)
    actions = get_reflective_actions()


    reflective_rank_request = RankRequest(actions=actions, context_features=context, event_id=rank_id_reflective)
    reflective_response = client.rank(rank_request=reflective_rank_request)
    reflective_ranked = reflective_response.reward_action_id

    patient = update_reflective_ranking(patient, reflective_ranked)
    print('RANKING RUN WITH CONTEXT :', context)

    rm_cols = ['num_pillsy_meds_t1', 'num_pillsy_meds_t2', 'num_pillsy_meds_t3', 'num_pillsy_meds_t4', 'num_pillsy_meds_t5', 'num_pillsy_meds_t6',
    'flag_send_reward_value_t0', 'reward_value_t0', 'rank_id_framing_t1' , 'rank_id_history_t1',  'rank_id_social_t1' ,  'rank_id_content_t1' ,
    'rank_id_reflective_t1' ,  'flag_send_reward_value_t1'  , 'reward_value_t1','adherence_day1','adherence_day2',  'adherence_day3',  'adherence_day4',
    'adherence_day5', 'adherence_day6', 'adherence_day7','dichot_adherence_day1', 'dichot_adherence_day2', 'dichot_adherence_day3', 'dichot_adherence_day4',
    'dichot_adherence_day5', 'dichot_adherence_day6', 'dichot_adherence_day7', 'total_dichot_adherence_past7', 'num_dates_early_rx_use', 'num_dates_disconnectedness',
    'num_days_continuously_disconnected',  'contact_disconnected', 'sms_msg_today', 'factor_set',  'text_number', 'text_message', 'framing_sms',
    'history_sms', 'social_sms',  'content_sms', 'reflective_sms',  'quantitative_sms',   'doctor_sms' , 'lifestyle_sms']
    pt_rank_log = patient.drop(rm_cols)


    #Framing
    if frame_response.ranking[0].id == 'posFrame':
        pt_rank_log['posFrame'] = frame_response.ranking[0].probability
    elif frame_response.ranking[0].id == 'negFrame':
        pt_rank_log['negFrame'] = frame_response.ranking[0].probability
    else:
        pt_rank_log['neutFram'] = frame_response.ranking[0].probability
    
    if frame_response.ranking[1].id == 'posFrame':
        pt_rank_log['posFrame'] = frame_response.ranking[1].probability
    elif frame_response.ranking[1].id == 'negFrame':
        pt_rank_log['negFrame'] = frame_response.ranking[1].probability
    else:
        pt_rank_log['neutFram'] = frame_response.ranking[1].probability

    if frame_response.ranking[2].id == 'posFrame':
        pt_rank_log['posFrame'] = frame_response.ranking[2].probability
    elif frame_response.ranking[2].id == 'negFrame':
        pt_rank_log['negFrame'] = frame_response.ranking[2].probability
    else:
        pt_rank_log['neutFram'] = frame_response.ranking[2].probability

    # History
    if history_response_flag:
        if history_response.ranking[0].id == 'yesHistory':               
            pt_rank_log['yesHistory'] = history_response.ranking[0].probability
        else: 
            pt_rank_log['noHistory'] = history_response.ranking[0].probability
        if history_response.ranking[1].id == 'yesHistory':               
            pt_rank_log['yesHistory'] = history_response.ranking[1].probability
        else: 
            pt_rank_log['noHistory'] = history_response.ranking[1].probability
    
    # Social
    if social_response.ranking[0].id == 'yesSocial':
        pt_rank_log['yesSocial'] = social_response.ranking[0].probability
    else: 
        pt_rank_log['noSocial'] = social_response.ranking[0].probability
    if social_response.ranking[1].id == 'yesSocial':
        pt_rank_log['yesSocial'] = social_response.ranking[1].probability
    else: 
        pt_rank_log['noSocial'] = social_response.ranking[1].probability

    # Content
    if content_response.ranking[0].id == 'yesContent':
        pt_rank_log['yesContent'] = content_response.ranking[0].probability
    else: 
        pt_rank_log['noContent'] = content_response.ranking[0].probability
    if content_response.ranking[1].id == 'yesContent':
        pt_rank_log['yesContent'] = content_response.ranking[1].probability
    else: 
        pt_rank_log['noContent'] = content_response.ranking[1].probability

    # Reflective
    if reflective_response.ranking[0].id == 'yesReflective':
        pt_rank_log['yesReflective'] = reflective_response.ranking[0].probability
    else: 
        pt_rank_log['noReflective'] = reflective_response.ranking[0].probability
    if reflective_response.ranking[1].id == 'yesReflective':
        pt_rank_log['yesReflective'] = reflective_response.ranking[1].probability
    else: 
        pt_rank_log['noReflective'] = reflective_response.ranking[1].probability
   

    print(pt_rank_log)

    patient = update_num_day_sms(patient)
    patient = updated_sms_today(patient)
    patient["trial_day_counter"] += 1
    return patient, pt_rank_log



def shift_t0_t1_rank_ids(patient):
    # shift these values for the next rank to store t0 values  
    patient["reward_value_t1"] = patient["reward_value_t0"]
    patient["flag_send_reward_value_t1"] = patient["flag_send_reward_value_t0"]
    patient["rank_id_framing_t1"] = patient["rank_id_framing_t0"]
    patient["rank_id_history_t1"] = patient["rank_id_history_t0"]
    patient["rank_id_social_t1"] = patient["rank_id_social_t0"]
    patient["rank_id_content_t1"] = patient["rank_id_content_t0"]
    patient["rank_id_reflective_t1"] = patient["rank_id_reflective_t0"]
    patient["reward_value_t0"] = 0
    patient["flag_send_reward_value_t0"] = False
    patient["rank_id_framing_t0"] = None
    patient["rank_id_history_t0"] = None
    patient["rank_id_social_t0"] = None
    patient["rank_id_content_t0"] = None
    patient["rank_id_reflective_t0"] = None
    return patient


def update_framing_ranking(patient, response_action_id_framing):
    patient["response_action_id_framing"] = response_action_id_framing
    if patient["response_action_id_framing"] == "posFrame":
        patient["framing_sms"] = 1
    elif patient["response_action_id_framing"] == "negFrame":
        patient["framing_sms"] = 2
    elif patient["response_action_id_framing"] == "neutFrame":
        patient["framing_sms"] = 0
    return patient

def update_history_ranking(patient, response_action_id_history):
    patient["response_action_id_history"] = response_action_id_history
    if patient["response_action_id_history"] == "yesHistory":
        patient["history_sms"] = 1
    elif patient["response_action_id_history"] == "noHistory":
        patient["history_sms"] = 0
    return patient

def update_social_ranking(patient, response_action_id_social):
    patient["response_action_id_social"] = response_action_id_social
    if patient["response_action_id_social"] == "yesSocial":
        patient["social_sms"] = 1
    elif patient["response_action_id_social"] == "noSocial":
        patient["social_sms"] = 0
    return patient

def update_content_ranking(patient, response_action_id_content):
    patient["response_action_id_content"] = response_action_id_content
    if patient["response_action_id_content"] == "yesContent":
        patient["content_sms"] = 1
    elif patient["response_action_id_content"] == "noContent":
        patient["content_sms"] = 0
    return patient

def update_reflective_ranking(patient, response_action_id_reflective):
    patient["response_action_id_reflective"] = response_action_id_reflective
    if patient["response_action_id_reflective"] == "yesReflective":
        patient["reflective_sms"] = 1
    elif patient["response_action_id_reflective"] == "noReflective":
        patient["reflective_sms"] = 0
    return patient

def update_num_day_sms(patient):
    if patient["response_action_id_framing"] == "posFrame":
        patient["num_day_since_pos_framing"] = 0
        patient["num_day_since_neg_framing"] += 1
        patient["num_day_since_no_sms"] = 0
    elif patient["response_action_id_framing"] == "negFrame":
        patient["num_day_since_neg_framing"] = 0
        patient["num_day_since_pos_framing"] += 1
        patient["num_day_since_no_sms"] = 0
    elif patient["response_action_id_framing"] == "neutFrame":
        patient["num_day_since_neg_framing"] += 1
        patient["num_day_since_pos_framing"] += 1

    if patient["response_action_id_history"] == "yesHistory":
        patient["num_day_since_history"] = 0
        patient["num_day_since_no_sms"] = 0
    elif patient["response_action_id_history"] == "noHistory":
        patient["num_day_since_history"] += 1

    if patient["response_action_id_social"] == "yesSocial":
        patient["num_day_since_social"] = 0
        patient["num_day_since_no_sms"] = 0
    elif patient["response_action_id_social"] == "noSocial":
        patient["num_day_since_social"] += 1

    if patient["response_action_id_content"] == "yesContent":
        patient["num_day_since_content"] = 0
        patient["num_day_since_no_sms"] = 0
    elif patient["response_action_id_content"] == "noContent":
        patient["num_day_since_content"] += 1

    if patient["response_action_id_reflective"] == "yesReflective":
        patient["num_day_since_reflective"] = 0
        patient["num_day_since_no_sms"] = 0
    elif patient["response_action_id_reflective"] == "noReflective":
        patient["num_day_since_reflective"] += 1

    if patient["response_action_id_framing"] == "neutFrame":
        if patient["response_action_id_history"] == "noHistory" and patient["response_action_id_social"] == "noSocial" and patient["response_action_id_content"] == "noContent" and patient["response_action_id_reflective"] == "noReflective":
            patient["num_day_since_no_sms"] += 1
    
    return patient

# Computes and updates the SMS text message to send to this patient today.
def updated_sms_today(patient):
    fp = build_path(os.path.abspath(os.curdir) + ("\\_SMSChoices"), "sms_choices.csv")
    sms_choices = pd.read_csv(fp)
    framing = patient["framing_sms"]
    history = patient["history_sms"]
    social = patient["social_sms"]
    content = patient["content_sms"]
    reflective = patient["reflective_sms"]
    print("records_id: ", patient["record_id"]," rankresult: ", framing, history, social, content, reflective)

    rows = sms_choices[sms_choices['framing_sms'] == framing]
    rows = rows[rows['history_sms'] == history]
    rows = rows[rows['social_sms'] == social]
    rows = rows[rows['content_sms'] == content]
    rows = rows[rows['reflective_sms'] == reflective]

  
    # If 0,0,0,0,0 is found, then the rows will be None, so our defaults are first, the empty text message
    text_number = 0
    factor_set = 0
    text = ""
    text_message = ""
    quantitative_sms = 0
    doctor_sms = 0
    lifestyle_sms = 0
    
    # If 0,0,0,0,0 is not found, then the rows will have some potential values,
    if not rows.empty:
        # Then we randomize what of the factor set text messages we will send
        row = rows.sample()
        # We record the factor_set and text_number as unique identifiers for this message
        factor_set = row['factor_set'].item()
        text_number = row['text_number'].item()
        quantitative_sms = row['quantitative_sms'].item()
        doctor_sms = row['doctor_sms'].item()
        lifestyle_sms = row['lifestyle_sms'].item()

        text_message = row['text_message'].item()
        # We store the text message that will be sent for this specific patient that takes into account the history of their adherence
        # This finds and replaces the "X" in the sms_choices text_message rows to customize to the patient.
        text = row['text_message'].item().replace("X", str(patient["total_dichot_adherence_past7"]))

    # We've updated the local variables and now store into the patient object as attributes to be exported in bulk by another function
    patient["text_number"] = text_number
    patient["factor_set"] = factor_set
    patient["text_message"] = text_message
    patient["quantitative_sms"] = quantitative_sms
    patient["doctor_sms"] = doctor_sms
    patient["lifestyle_sms"] = lifestyle_sms
    patient["sms_msg_today"] = text
    return patient

def get_demographics_features(patient):
    demographic_features = {"age": patient["age"],
                            "sex": patient["sex"],
                            "race_white": patient["race_white"],
                            "race_black": patient["race_black"],
                            "race_asian": patient["race_asian"],
                            "race_hispanic": patient["race_hispanic"],
                            "race_other": patient["race_other"],
                            "education_level": patient["edu_level"],
                            "employment_status": patient["employment_status"],
                            "marital_status": patient["marital_status"]}
    demographic_features_dict = {"demographic_features": demographic_features}
    return demographic_features_dict


def get_clinical_features(patient):
    clinical_features = {"num_physicians": patient["num_physicians"],
                         "num_years_dm_rx": patient["num_years_dm_rx"],
                         "hba1c": patient["hba1c"]}
    clinical_features_dict = {"clinical_features": clinical_features}
    return clinical_features_dict


def get_motivational_features(patient):
    motivational_features = {"automaticity": patient["automaticity"],
                             "pt_activation": patient["pt_activation"],
                             "reason_dm_rx": patient["reason_dm_rx"]}
    motivational_features_dict = {"motivational_features": motivational_features}
    return motivational_features_dict


def get_rx_use_features(patient):
    rx_use = {"num_rx": patient["num_rx"],
              "concomitant_insulin_use": patient["concomitant_insulin_use"],
              "non_adherence": patient["non_adherence"]}
    rx_use_dict = {"rx_use": rx_use}
    return rx_use_dict


def get_pillsy_med_features(patient):
    pillsy_med_features = {"num_twice_daily_pillsy_meds": patient["num_twice_daily_pillsy_meds"],
                           "pillsy_meds_agi": patient["pillsy_meds_agi"],
                           "pillsy_meds_dpp4": patient["pillsy_meds_dpp4"],
                           "pillsy_meds_glp1": patient["pillsy_meds_glp1"],
                           "pillsy_meds_meglitinide": patient["pillsy_meds_meglitinide"],
                           "pillsy_meds_metformin": patient["pillsy_meds_metformin"],
                           "pillsy_meds_sglt2": patient["pillsy_meds_sglt2"],
                           "pillsy_meds_sulfonylurea": patient["pillsy_meds_sulfonylurea"],
                           "pillsy_meds_thiazolidinedione": patient["pillsy_meds_thiazolidinedione"],
                           "num_pillsy_meds": patient["num_pillsy_meds_t0"]}
    pillsy_med_features_dict = {"pillsy_med_features": pillsy_med_features}
    return pillsy_med_features_dict


def get_observed_feedback_features(patient):
    observed_feedback_features = {}
    if patient["disconnectedness"] != None and patient["trial_day_counter"] >= 1:
        observed_feedback_features["disconnectedness"] = patient["disconnectedness"]
    if patient["early_rx_use"] != None and patient["trial_day_counter"] >= 1:
        observed_feedback_features["early_rx_use"] = patient["early_rx_use"]
    if (patient["avg_adherence_1day"] != None) and patient["trial_day_counter"] >= 1:
        observed_feedback_features["avg_adherence_1day"] = patient["avg_adherence_1day"]
    if (patient["avg_adherence_3day"] != None) and patient["trial_day_counter"] >= 3:
        observed_feedback_features["avg_adherence_3day"] = patient["avg_adherence_3day"]
    if (patient["avg_adherence_7day"] != None) and patient["trial_day_counter"] >= 7:
        observed_feedback_features["avg_adherence_7day"] = patient["avg_adherence_7day"]
    observed_feedback_features_dict = {"observed_feedback_features": observed_feedback_features}
    return observed_feedback_features_dict



def get_num_days_since_features(patient):
    num_days_since_features = {"num_day_since_no_sms": patient["num_day_since_no_sms"],
                               "num_day_since_pos_framing": patient["num_day_since_pos_framing"],
                               "num_day_since_neg_framing": patient["num_day_since_neg_framing"],
                               "num_day_since_history": patient["num_day_since_history"],
                               "num_day_since_social": patient["num_day_since_social"],
                               "num_day_since_content": patient["num_day_since_content"],
                               "num_day_since_reflective": patient["num_day_since_reflective"]}
    num_days_since_features_dict = {"num_days_since_features": num_days_since_features}
    return num_days_since_features_dict


def get_framing_context(patient):
    framing_context = [
        get_demographics_features(patient),
        get_clinical_features(patient),
        get_motivational_features(patient),
        get_rx_use_features(patient),
        get_pillsy_med_features(patient),
        get_observed_feedback_features(patient),
        get_num_days_since_features(patient)]
    return framing_context


def get_history_context(patient):
    history_context = [
        get_demographics_features(patient),
        get_clinical_features(patient),
        get_motivational_features(patient),
        get_rx_use_features(patient),
        get_pillsy_med_features(patient),
        get_observed_feedback_features(patient),
        get_num_days_since_features(patient),
        {"response_action_id_framing" : patient["response_action_id_framing"]}]
    return history_context


def get_social_context(patient):
    social_context = [
        get_demographics_features(patient),
        get_clinical_features(patient),
        get_motivational_features(patient),
        get_rx_use_features(patient),
        get_pillsy_med_features(patient),
        get_observed_feedback_features(patient),
        get_num_days_since_features(patient),
        {"response_action_id_framing" : patient["response_action_id_framing"]},
        {"response_action_id_history" : patient["response_action_id_history"]}]
    return social_context


def get_content_context(patient):
    content_context = [
        get_demographics_features(patient),
        get_clinical_features(patient),
        get_motivational_features(patient),
        get_rx_use_features(patient),
        get_pillsy_med_features(patient),
        get_observed_feedback_features(patient),
        get_num_days_since_features(patient),
        {"response_action_id_framing" : patient["response_action_id_framing"]},
        {"response_action_id_history" : patient["response_action_id_history"]},
        {"response_action_id_social" : patient["response_action_id_social"]}]
    return content_context


def get_reflective_context(patient):
    reflective_context = [
        get_demographics_features(patient),
        get_clinical_features(patient),
        get_motivational_features(patient),
        get_rx_use_features(patient),
        get_pillsy_med_features(patient),
        get_observed_feedback_features(patient),
        get_num_days_since_features(patient),
        {"response_action_id_framing" : patient["response_action_id_framing"]},
        {"response_action_id_history" : patient["response_action_id_history"]},
        {"response_action_id_social" : patient["response_action_id_social"]},
        {"response_action_id_content" : patient["response_action_id_content"]}]
    return reflective_context


All of the following are drawn from `driverReward.py`, one of the main features of the workflow.

In [ ]:
# <reward>
import sys
import time
from azure.cognitiveservices.personalizer import PersonalizerClient
from azure.cognitiveservices.personalizer.models import RankRequest
from msrest.authentication import CognitiveServicesCredentials
import pandas as pd
import numpy as np
import math
import time
from datetime import datetime, date, timedelta
#import pytz
from collections import Counter
import string
import pickle
import json
import os
from datetime import date
import http.client, urllib.request, urllib.parse, urllib.error, base64

from exe_functions import build_path

def get_reward_update(pt_data, run_time):
    fp = build_path("000_RewardData", str(run_time.date()) + "_reward_updates.csv")
    today = run_time.date()
    two_day_ago = (run_time - timedelta(days=2)).date()
    yesterday = (run_time - timedelta(days=1)).date()


    # Subset updated_pt_dict to what we need for reward calls and put in dataframe
    # create an Empty DataFrame object
    column_values = ['reward', 'frame_id', 'history_id', 'social_id', 'content_id', 'reflective_id', 'record_id', 'trial_day_counter', 
                     'flag_send_reward_value_tX']
    reward_updates = pd.DataFrame(columns=column_values)

    for pt,data_row in pt_data.iterrows():
        # Reward value, Rank_Id's
        if(data_row["flag_send_reward_value_t0"] == True and data_row["censor_date"] >= today):
            reward_row_t0 = [data_row["reward_value_t0"], data_row["rank_id_framing_t0"], data_row["rank_id_history_t0"],
                       data_row["rank_id_social_t0"], data_row["rank_id_content_t0"], data_row["rank_id_reflective_t0"],
                       data_row["record_id"], data_row["trial_day_counter"], "flag_send_reward_value_t0"]
            reward_updates.loc[len(reward_updates)] = reward_row_t0
        if(data_row["flag_send_reward_value_t1"] == True and data_row["censor_date"] >= yesterday):
            reward_row_t1 = [data_row["reward_value_t1"], data_row["rank_id_framing_t1"], data_row["rank_id_history_t1"],
                       data_row["rank_id_social_t1"], data_row["rank_id_content_t1"], data_row["rank_id_reflective_t1"],
                       data_row["record_id"], data_row["trial_day_counter"], "flag_send_reward_value_t1"]
            reward_updates.loc[len(reward_updates)] = reward_row_t1
       
    # Write csv as a log for what we're sending to Personalizer
    reward_updates.to_csv(fp, index=False)
    reward_updates = reward_updates.to_numpy()
    return reward_updates


def send_rewards(reward_updates, client):
    # column_values = ['reward', '
    #   frame_id', 'history_id', 'social_id', 'content_id', 'reflective_id',
    #   'study_id', 'trial_day_counter']
    for i in range(0,reward_updates.shape[0]):
        row = reward_updates[i, :]
        reward_val = row[0] 
        for j in range(1,6):
            if isinstance(row[j],str):
                print("reward_val: ", reward_val)
                print("event_id: ", row[j])
                client.events.reward(event_id=row[j], value=reward_val)
            

            ##############---- If checking for connection with Personalizer ###############
            # headers = {
            #     # Request headers
            #     'Content-Type': 'application/json-patch+json',
            #     'Ocp-Apim-Subscription-Key': '{subscription key}',
            # }

            # params = urllib.parse.urlencode({
            # })

            # try:
            #     conn = http.client.HTTPSConnection('westus2.api.cognitive.microsoft.com')
            #     conn.request("POST", "/personalizer/v1.0/events/{eventId}/reward?%s" % params, "{body}", headers)
            #     response = conn.getresponse()
            #     data = response.read()
            #     print(data)
            #     conn.close()
            # except Exception as e:
            #     print("[Errno {0}] {1}".format(e.errno, e.strerror))

            ################################################################################


All of the following are drawn from `RL_Personalizer.py` this appears to be the core program that gets ported over to the `.exe`

In [ ]:
#!/usr/bin/env python
# coding: utf-8

import sys
import time
import dateutil
import dateutil.parser
from azure.cognitiveservices.personalizer import PersonalizerClient
from azure.cognitiveservices.personalizer.models import RankRequest
from msrest.authentication import CognitiveServicesCredentials
import pandas as pd
import numpy as np
import math
import time
from datetime import datetime, timedelta
from collections import Counter
import string
import pickle
import json
#import pytz
import os
import re

from patient_data import import_pt_data, new_empty_pt_data
from pillsy_parser import import_Pillsy, find_rewards
from driverReward import get_reward_update,send_rewards
from redcap_parser import import_redcap, update_pt_data_with_redcap
from driverRank import run_ranking, write_sms_history, new_empty_rank_log, write_rank_log
from control_disconnection import check_control_disconnectedness, import_redcap_control, import_pt_data_control
from exe_functions import build_path


## 1. Get date

############# For real deal #############
run_time = datetime.now()
## REQUIRES REVIEW AND pytz WILL NEED TO BE REMOVED
run_time = pytz.timezone("America/New_York").localize(run_time)
#########################################

############## For testing ##############
# while True:
#     try:
#         run_time = datetime.strptime(input("Enter YYYY-MM-DD testing date (time will be 9:30 AM): "), "%Y-%m-%d")
#         if run_time.year in [2020, 2021]:
#             run_time = run_time + timedelta(hours=9, minutes=30)
#             break
#         else:
#             print("testing year must be 2020 or 2021")
#     except ValueError as ve:
#         print(ve)
# # run_time = pytz.timezone("America/New_York").localize(dateutil.parser.parse("10:30 AM 2020-12-15"))
# run_time = pytz.timezone("America/New_York").localize(run_time)
# print("successful date entered\nrun_time: {}".format(run_time))
#########################################

## 2. Check if program has been run today

fp = build_path("_ProgramLog", str(run_time.date()) + "_RL_Personalizer_log.txt")
if os.path.isfile(fp): 
    input("\nALREADY RAN TODAY: {}.\n".format(run_time.strftime("%B %d, %Y"))
          + "Please contact the other RAs to confirm someone else has already run it today.\n"
          + "Press Enter to exit the program and close this window.")
    sys.exit()

## 3. Ask if first day of trial 

# Embedded this into the import_pt_data, import_pt_data_control, import_Pillsy functions
# while True:
#     first_day = input("\nIs today the trial initiation?\n" 
#                       + "If today is the first day, type 'yes' then hit Enter.\n"
#                       + "Otherwise type 'no' then hit Enter.\n"
#                       + "Answer here: ").lower()
#     if first_day in ["yes", "no"]:
#         first_day = first_day == "yes"
#         break
#     else:
#         print("Input was not 'yes' or 'no'. Please try again.")

## 4. Check for (non)existence of files

pt_data = import_pt_data(run_time)
pt_data_control = import_pt_data_control(run_time)
new_pillsy_data = import_Pillsy(run_time)
redcap_data = import_redcap(run_time)
redcap_control = import_redcap_control(run_time)

## 5. Start log for program

old_stdout = sys.stdout        
log_file = open(fp, "w")
sys.stdout = log_file

## 6. Start main body of program

print("-----------------------------BEGIN PROGRAM----------------------------")
print(run_time)

## Set Up MS Azure Personalizer Client
print("-----------------------------CREATE PERSONALIZER CLIENT----------------------------")
with open(build_path(".keys", "azure-personalizer-key.txt"), 'r') as f:
     personalizer_key = f.read().rstrip()
client = PersonalizerClient(
    "https://bwh-pharmacoepi-roybal-dev-use2-cog.cognitiveservices.azure.com/", 
    CognitiveServicesCredentials(personalizer_key)
)


# ## Reward Step
# 
# If we've already initiated the trial, we will have:
# * Pre-existing patient dataset in need of reward updates
# * Pillsy data from yesterday to determine reward
# If this is study initiation, this step will just load an empty patient dictionary and null pillsy dataset.


if not pt_data.empty and new_pillsy_data is not None:
    print("----------------------------IMPORT PILLSY AND PT DATA SUCCESS---------------------------")
    print("----------------------------------RUNNING FIND REWARDS----------------------------------")
    # From Pillsy data, computes the Rewards to send to Personalizer for each patient's Rank calls from yesterday's run.
    pt_data = find_rewards(new_pillsy_data, pt_data, run_time)
    print("---------------------------FORMATTING REWARDS FOR PERSONALIZER--------------------------")
    # using updated patient data (new pillsy + patient data), format the rewards to Personalizer into a dataframe
    rewards_to_send = get_reward_update(pt_data, run_time)
    print("-----------------------------SENDING REWARDS TO PERSONALIZER----------------------------")
    # actual call to personalizer
    send_rewards(rewards_to_send, client)


# ## Import/Update Patients

print("-----------------------------IMPORT REDCAP AND PT DATA----------------------------")
pt_data = update_pt_data_with_redcap(redcap_data, pt_data, run_time)

# ## Rank Step
# Call Personalizer to rank action features to find the correct text message to send today.

ranked_pt_data = new_empty_pt_data()
ranking_log = new_empty_rank_log(run_time)
print("---------------------------------RANKING PATIENTS---------------------------------")
for index, patient in pt_data.iterrows():
    if patient["censor"] != 1 and patient["censor_date"] > run_time.date():
        patient, pt_rank_log = run_ranking(patient, client, run_time)
        ranked_pt_data = ranked_pt_data.append(patient)
        ranking_log = ranking_log.append(pt_rank_log)

print("---------------------------------EXPORT RANK LOG FILE-----------------------------")
write_rank_log(ranking_log, run_time)
# ## Output SMS and Patient Data

print("---------------------------------CHECKING CONTROLS---------------------------------")
check_control_disconnectedness(new_pillsy_data,redcap_control,pt_data_control,run_time) # check whether controls have connection problems

print("---------------------------------EXPORT SMS FILE----------------------------------")
write_sms_history(ranked_pt_data, run_time)
ranked_pt_data.to_csv(
    build_path("000_PatientData", str(run_time.date()) + "_pt_data.csv"), 
    index=False
)

print("-----------------------------------------------------------------------------------")
log_file.close()
sys.stdout = old_stdout

print("---------------------------------PROGRAM SUCCESSFULLY RAN--------------------------")
input("SUCCESSFULLY RAN TODAY: {} \n".format(run_time.strftime("%B %d, %Y"))
        + "Now, send messages to patients from /000_SMS_TO_SEND/" + str(run_time.date()) + "_sms_history.csv"
        + "\nPress Enter to exit the program and close this window.")
sys.exit()
